In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import PolynomialFeatures,LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split

# Lets start by Loading the data and checking info of it with columns

In [2]:
df = pd.read_csv('crop_yield_polynomial_regression.csv')

print(f"Dataset shape: {df.shape}")
print(f"\nColumn info:\n{df.dtypes}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nFirst 5 rows:\n{df.head()}")
print(f"\nDescriptive statistics:\n{df.describe()}")


Dataset shape: (500, 5)

Column info:
CropType                         str
Fertilizer_kg_per_hectare    float64
Water_mm_per_season          float64
Pesticide_L_per_hectare      float64
CropYield_tons               float64
dtype: object

Missing values:
CropType                      0
Fertilizer_kg_per_hectare    17
Water_mm_per_season          15
Pesticide_L_per_hectare      12
CropYield_tons                8
dtype: int64

First 5 rows:
  CropType  Fertilizer_kg_per_hectare  Water_mm_per_season  \
0     Corn                      134.7                530.6   
1  Soybean                      108.1                798.5   
2    Wheat                      145.5                822.8   
3    Wheat                      112.2                599.0   
4  Soybean                      179.5                745.8   

   Pesticide_L_per_hectare  CropYield_tons  
0                     5.75            9.17  
1                     6.18            6.44  
2                     8.60            6.41  
3    

# Above describe() show us that Column names are acceptable because there's no spacing in them but let's get them in lower()
1. Null values in different columns.
2. negative values in all columns.
3. Most dtypes are float so no need to change to numbers
4. CropYield_tons has 8 nulls drop them 


In [8]:
print(f"Original Number of Rows: {len(df)}")

# --- 3a: Standardize Column Names ---
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
for _ in range(3):
    df.columns = df.columns.str.replace('__', '_')

print("\nMissing values before cleaning:")
print(df.isnull().sum())

# --- 3b: Filter Impossible Physical Values ---
impossible_mask = (
    (df['fertilizer_kg_per_hectare'] < 0) | 
    (df['water_mm_per_season'] < 0) | 
    (df['pesticide_l_per_hectare'] < 0) | 
    (df['cropyield_tons'] < 0) | 
    (df['fertilizer_kg_per_hectare'] > 1000) | 
    (df['pesticide_l_per_hectare'] > 50) | 
    (df['water_mm_per_season'] > 5000)
)
df_clean = df[~impossible_mask].copy()

# --- 3c: Handle Missing Values (Warning-Free Direct Assignment) ---
num_cols = ['fertilizer_kg_per_hectare', 'water_mm_per_season', 'pesticide_l_per_hectare']
for col in num_cols:
    df_clean[col] = df_clean[col].fillna(df_clean[col].median())

# Drop rows missing the target variable
df_clean = df_clean.dropna(subset=['cropyield_tons'])

# --- 3d: Categorical Encoding ---
le = LabelEncoder()
df_clean['croptype_encoded'] = le.fit_transform(df_clean['croptype']) # type: ignore

print(f"\nCleaned dataset shape: {df_clean.shape}")
print(f"Remaining rows: {len(df_clean)}")

print("\nEncoded categories mapping:")
for i, item in enumerate(le.classes_):
    print(f"  {item} -> {i}")

Original Number of Rows: 500

Missing values before cleaning:
croptype                      0
fertilizer_kg_per_hectare    17
water_mm_per_season          15
pesticide_l_per_hectare      12
cropyield_tons                8
dtype: int64

Cleaned dataset shape: (485, 6)
Remaining rows: 485

Encoded categories mapping:
  Corn -> 0
  Rice -> 1
  Soybean -> 2
  Wheat -> 3


In [10]:
df_clean.isnull().sum()

croptype                     0
fertilizer_kg_per_hectare    0
water_mm_per_season          0
pesticide_l_per_hectare      0
cropyield_tons               0
croptype_encoded             0
dtype: int64

In [12]:
print("\n--- Correlation with cropyield_tons ---")
print(df_clean.select_dtypes(include='number').corr()['cropyield_tons'].sort_values(ascending=False))


--- Correlation with cropyield_tons ---
cropyield_tons               1.000000
fertilizer_kg_per_hectare    0.047568
water_mm_per_season         -0.006581
pesticide_l_per_hectare     -0.128069
croptype_encoded            -0.615228
Name: cropyield_tons, dtype: float64
